# MERFISH onto Visium H&E, from annotated curves

Landmarks traced as curves rather than clicked as points. A curve is a correspondence between
two *shapes*, not between two lists of vertices, and that distinction is the whole content of
this notebook.

**Upstream's own version of this notebook does not run.** Its two saved curve files hold 10 and
15 vertices, so `L_T_from_points` raises `Number of pointsI (10) is not equal to number of
pointsJ (15)` -- and upstream's committed output records that same exception. Squidpy raises
too, and for the same good reason: paired landmarks are matched by row, so unequal counts have
no meaning. The fix belongs to the caller, and it is to resample.

## Inputs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, spatialdata as sd
from spatialdata.models import Image2DModel, PointsModel
from squidpy.experimental.im import rasterize_points
from squidpy.experimental.tl import align_stalign_image

def as_image(rgb, key):
    return sd.SpatialData(images={key: Image2DModel.parse(
        np.moveaxis(rgb, -1, 0).astype(float), dims=('c', 'y', 'x'))})

def rasterized(xy, dx):
    sdata = sd.SpatialData(points={'cells': PointsModel.parse(xy)})
    rasterize_points(sdata, 'cells', dx=dx, blur=1.0, key_added='section')
    return sdata

MERFISH = ('merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase'
           '_Slice2_Replicate3_cell_metadata_S2R3.csv.gz')
cells = pd.read_csv(MERFISH)
xy = np.c_[cells['center_x'], cells['center_y']].astype(float)

he = plt.imread('visium_data/tissue_hires_image.png')[..., :3]
visium = as_image(he, 'he')
merfish = rasterized(xy, 30.0)

traced = {side: np.load(f'visium_data/{name}_curves.npy', allow_pickle=True).item()
          for side, name in (('query', 'Merfish_S2_R3'), ('ref', 'tissue_hires_image'))}
for name in traced['ref']:
    print(f'{name:<8} ref {len(traced["ref"][name]):>3} vertices, '
          f'query {len(traced["query"][name]):>3}')

## Resampling the curves

Each curve is resampled to the same number of points, evenly along its own arc length. That is
what makes the two sides comparable: vertex *k* of one curve then corresponds to vertex *k* of
the other because both are the same fraction along the shape, which is not true of whatever
vertex count the annotator happened to record.

In [ ]:
def resampled(curve, n):
    points = np.asarray(curve, dtype=float)
    walked = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(points, axis=0), axis=1))]
    even = np.linspace(0.0, walked[-1], n)
    return np.c_[np.interp(even, walked, points[:, 0]), np.interp(even, walked, points[:, 1])]

PER_CURVE = 8
paired = {side: np.vstack([resampled(traced[side][name], PER_CURVE) for name in traced['ref']])
          for side in traced}
print(f'{len(paired["ref"])} pairs from {len(traced["ref"])} curves at {PER_CURVE} points each')

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].imshow(he); ax[0].scatter(*paired['ref'].T, s=12, c='red')
ax[0].set_title('Visium H&E, curves resampled')
ax[1].scatter(*xy.T, s=0.12, alpha=0.3); ax[1].scatter(*paired['query'].T, s=12, c='red')
ax[1].set_title('MERFISH section, curves resampled')
ax[1].invert_yaxis(); ax[1].set_aspect('equal')
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## The fit

In [ ]:
fit = align_stalign_image(
    visium, merfish, image_key=('he', 'section'),
    landmarks_ref=paired['ref'], landmarks_query=paired['query'],
    niter=200, sigmaM=0.18, sigmaB=0.18, sigmaA=0.18, sigmaP=2e-1,
    epL=5e-11, epT=5e-4, epV=5e1,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

## Every cell, placed on the image

In [ ]:
placed = np.asarray(fit.transform(xy))
residual = np.linalg.norm(np.asarray(fit.transform(paired['query'])) - paired['ref'], axis=1)
rows, columns = he.shape[:2]
inside = ((placed[:, 0] >= 0) & (placed[:, 0] < columns)
          & (placed[:, 1] >= 0) & (placed[:, 1] < rows))
print(f'landmark residual: median {np.median(residual):.1f} px, worst {residual.max():.1f} px')
print(f'{100 * inside.mean():.0f}% of cells land within the {columns} x {rows} image')

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(he); ax[0].scatter(*paired['ref'].T, s=12, c='red')
ax[0].set_title('Visium H&E')
ax[1].imshow(he); ax[1].scatter(*placed.T, s=0.12, alpha=0.3, c='tab:blue')
ax[1].set_title('MERFISH cells placed on it')
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## The objective's trace

In [ ]:
MIXTURE_GATE = 50
energies = np.asarray(fit.energies)[: fit.n_iter]
descent = energies[MIXTURE_GATE + 1 :]
tail = descent[-max(len(descent) // 10, 1) :]
print(f'after the gate: {descent[0]:.0f} -> {descent[-1]:.0f}, minimum {descent.min():.0f} '
      f'at iteration {MIXTURE_GATE + 1 + int(descent.argmin())}')
print(f'last tenth: mean {tail.mean():.0f}, spread {np.ptp(tail):.0f} '
      f'({100 * np.ptp(tail) / tail.mean():.1f}% of its mean)')
plt.plot(energies, lw=0.8); plt.axvline(MIXTURE_GATE, color='0.6', ls='--', lw=0.8)
plt.xlabel('iteration'); plt.ylabel('objective'); plt.grid(alpha=0.3)